In [ ]:

# ── 1. Environment Setup ──────────────────────────────────────────────────────
!git clone https://github.com/goombalab/hnet.git
!git clone --recurse-submodules https://github.com/Chaoqi-LIU/oat.git
!rm -rf congenial-adventure && git clone https://github.com/SDogya/congenial-adventure.git && cp -r congenial-adventure/. . && rm -rf congenial-adventure

import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
os.environ['WANDB_API_KEY'] = _s.get_secret('wandb')

!uv add "zarr<3.0.0" dill einops numba vector-quantize-pytorch accelerate \
         huggingface_hub "robomimic<0.3.0" torchvision wrapt pillow pandas \
         diffusers av gymnasium libero
!uv sync

# Patch OAT lr_scheduler.py — newer diffusers removed Union/Optional/Optimizer exports
p = 'oat/oat/model/common/lr_scheduler.py'
txt = open(p).read()
marker = 'from diffusers.optimization import ('
if marker in txt and 'from typing import Union' not in txt:
    idx = txt.index(marker)
    end_idx = txt.index(')', idx) + 1
    header = (
        'from typing import Union, Optional\n'
        'from torch.optim import Optimizer\n'
        'from diffusers.optimization import SchedulerType, TYPE_TO_SCHEDULER_FUNCTION'
    )
    open(p, 'w').write(txt[:idx] + header + txt[end_idx:])
    print('lr_scheduler.py patched OK')
else:
    print('lr_scheduler.py already clean')

# Suppress noisy normalizer prints from OAT encoders
for path in [
    'oat/oat/perception/state_encoder.py',
    'oat/oat/perception/robomimic_vision_encoder.py',
]:
    txt = open(path).read()
    txt = txt.replace(
        'print(warning_msg(f"no normalizer params for port {port}, skipping normalization."))',
        'pass  # suppressed'
    ).replace(
        'print(f"no normalizer params for port {port}, skipping normalization.")',
        'pass  # suppressed'
    )
    open(path, 'w').write(txt)
print('normalizer prints suppressed OK')


In [ ]:

# ── 2. Dataset ────────────────────────────────────────────────────────────────
import os
from huggingface_hub import login, snapshot_download
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret('hugface'))

os.makedirs('/kaggle/working/oat/data/libero', exist_ok=True)
snapshot_download(
    repo_id='chaoqi-liu/libero10_N500.zarr',
    repo_type='dataset',
    local_dir='/kaggle/working/oat/data/libero'
)

!unzip -o -q /kaggle/working/oat/data/libero/libero10_N500.zarr.zip \
             -d /kaggle/working/oat/data/libero/
!ls -lh /kaggle/working/oat/data/libero/


In [ ]:
# ── 3. Train OAT Tokenizer (~2-3h, 300 epochs) ───────────────────────────────
!uv run python oat/scripts/run_workspace.py \
    --config-name=train_oattok \
    task/tokenizer=libero/libero10 \
    training.num_epochs=300 \
    logging.project=VLA-experiment \
    task.tokenizer.dataset.zarr_path="/kaggle/working/oat/data/libero/libero10_N500.zarr"

In [ ]:
# ── 4. Train FD-DRAT ─────────────────────────────────────────────────────────
# model.H_l=8 must match the OAT tokenizer's latent_horizon.
# Without this override the config default (64) causes nested dropout
# to be almost never applied during training (87% of K samples exceed 8).
TOK = 'src/model/model.ckpt'
print(f'Using tokenizer: {TOK}')

!HYDRA_FULL_ERROR=1 MPLBACKEND=agg uv run run.py \
    strategy=single_gpu \
    model.tokenizer_ckpt={TOK} \
    model.H_l=8 \
    dataset_path=/kaggle/working/oat/data/libero/libero10_N500.zarr \
    batch_size=16

In [ ]:
# ── 5. Evaluation ─────────────────────────────────────────────────────────────
import os, subprocess, pathlib, yaml

# EGL headless rendering — native MuJoCo EGL binding, no PyOpenGL needed.
# osmesa fails on Kaggle because libOpenGL.so is not found by PyOpenGL.
os.environ['MUJOCO_GL']  = 'egl'
os.environ.pop('PYOPENGL_PLATFORM', None)  # must NOT be set for native EGL
os.environ['MPLBACKEND'] = 'agg'           # Jupyter inline backend not in uv env

subprocess.run(['apt-get', 'install', '-y', '-q', 'libegl1'], check=False)

# Pre-create LIBERO config to prevent interactive prompt
_cfg_dir  = pathlib.Path.home() / '.libero'
_cfg_dir.mkdir(exist_ok=True)
_cfg_file = _cfg_dir / 'config.yaml'
if not _cfg_file.exists():
    _pkg = subprocess.check_output(
        ['uv', 'run', 'python', '-c',
         'import libero.libero, pathlib; print(pathlib.Path(libero.libero.__file__).parent)'],
        text=True
    ).strip()
    _pkg = pathlib.Path(_pkg)
    yaml.dump({
        'benchmark_root': str(_pkg),
        'bddl_files':     str(_pkg / 'bddl_files'),
        'init_states':    str(_pkg / 'init_files'),
        'datasets':       str(_pkg.parent / 'datasets'),
        'assets':         str(_pkg / 'assets'),
    }, _cfg_file.open('w'))
    print(f'Created LIBERO config → {_cfg_file}')
else:
    print(f'LIBERO config already exists: {_cfg_file}')

# n_parallel_envs=10 → ceil(50/10)=5 batches, ~18GB RAM.
# Increase if RAM headroom allows; decrease if OOM.
!MPLBACKEND=agg uv run python scripts/eval_fddrat_libero.py \
    -c src/model/eval_mod.ckpt \
    -o eval_out/ \
    --n_test 50 \
    --n_test_vis 5

In [ ]:
# ── 6. Results Visualization ──────────────────────────────────────────────────
import json, pathlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

log = json.load(open('eval_out/eval_log.json'))

task_keys  = sorted([k for k in log if k.endswith('/mean_success_rate_mean')])
task_names = [k.replace('/mean_success_rate_mean', '').split('_SCENE')[0] for k in task_keys]
task_names = [t.replace('LIBERO_', '').replace('_', ' ').title() for t in task_names]
task_vals  = [log[k] for k in task_keys]
task_errs  = [log.get(k.replace('_mean', '_stderr'), 0.0) for k in task_keys]

mean_sr  = log.get('mean_success_rate_mean', np.mean(task_vals))
mean_err = log.get('mean_success_rate_stderr', 0.0)
lat_mean = log.get('latency_mean_ms', float('nan'))
lat_p99  = log.get('latency_p99_ms',  float('nan'))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('FD-DRAT — LIBERO10 Evaluation', fontsize=14, fontweight='bold')

ax = axes[0]
colors = ['#2ecc71' if v >= 0.8 else '#e67e22' if v >= 0.5 else '#e74c3c' for v in task_vals]
bars = ax.barh(task_names, task_vals, xerr=task_errs, color=colors,
               capsize=4, edgecolor='white', height=0.6)
ax.axvline(mean_sr, color='#3498db', linestyle='--', linewidth=1.5,
           label=f'Mean {mean_sr:.1%}')
ax.set_xlim(0, 1.05)
ax.set_xlabel('Success Rate')
ax.set_title('Per-Task Success Rate')
ax.legend(handles=[
    mpatches.Patch(color='#2ecc71', label='≥ 80%'),
    mpatches.Patch(color='#e67e22', label='50–80%'),
    mpatches.Patch(color='#e74c3c', label='< 50%'),
], fontsize=8, loc='lower right')
for bar, val in zip(bars, task_vals):
    ax.text(min(val + 0.01, 1.0), bar.get_y() + bar.get_height() / 2,
            f'{val:.0%}', va='center', fontsize=8)

ax2 = axes[1]
ax2.axis('off')
rows = [
    ['Metric', 'Value'],
    ['Mean Success Rate',   f'{mean_sr:.1%} ± {mean_err:.1%}'],
    ['Tasks ≥ 80%',        f'{sum(v >= 0.8 for v in task_vals)} / {len(task_vals)}'],
    ['Tasks ≥ 50%',        f'{sum(v >= 0.5 for v in task_vals)} / {len(task_vals)}'],
    ['Latency mean',       f'{lat_mean:.1f} ms'],
    ['Latency p99',        f'{lat_p99:.1f} ms'],
    ['Num experiments',    str(log.get('num_exp', 1))],
]
tbl = ax2.table(cellText=rows[1:], colLabels=rows[0],
                cellLoc='center', loc='center', bbox=[0.05, 0.1, 0.9, 0.8])
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#ecf0f1')
ax2.set_title('Summary', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('eval_out/results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nMean success rate: {mean_sr:.1%}  |  p99 latency: {lat_p99:.1f} ms')

In [ ]:
# ── 7. Export Results ─────────────────────────────────────────────────────────
import shutil
shutil.make_archive('eval_results', 'zip', 'eval_out/')
print('eval_results.zip ready — download from Output tab')